In [ ]:
# CYR-GPU-012 / R1 — CELL 0: FREEZE / VERIFY / CALIBRATE / RESOLVE
import json, os, subprocess, sys
from pathlib import Path
REPO = Path('/content/An-Ra-the-new-AGI')
BRANCH = 'cymek-500m-readiness'
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
P = REPO/'docs/cymek/experiments/CYR-GPU-012-R1/PREREGISTRATION.json'
R = REPO/'docs/cymek/experiments/CYR-GPU-012-R1/RUN_READINESS.json'
if not P.exists() or not R.exists(): raise RuntimeError('R1 is not frozen/readiness-bound')
PREREG=json.loads(P.read_text()); READINESS=json.loads(R.read_text())
if READINESS.get('ready_for_operator_colab_gpu_run') is not True: raise RuntimeError('R1 readiness is not green')
if READINESS.get('executable_sha') != PREREG.get('executable_sha'): raise RuntimeError('readiness/prereg SHA mismatch')
Path('/content/CYR-GPU-012-R1-PREREGISTRATION.json').write_text(json.dumps(PREREG,indent=2,sort_keys=True))
EXECUTABLE_SHA=PREREG['executable_sha']
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXECUTABLE_SHA],check=True)
HEAD=subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
assert HEAD==EXECUTABLE_SHA
for relative, expected_blob in PREREG['executable_blobs'].items():
    actual=subprocess.run(['git','-C',str(REPO),'hash-object',relative],check=True,capture_output=True,text=True).stdout.strip()
    assert actual==expected_blob, f'blob mismatch: {relative}'
print('R1 frozen executable verified:',HEAD)
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers','pytest','numpy'],check=True)
sys.path.insert(0,str(REPO))
subprocess.run([sys.executable,'-m','py_compile','v5_experiments/cyr_gpu012_r1.py','anra_v5/cyr_gpu012_r1_run.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_v5_cyr_gpu012_r1.py','tests/test_v5_cyr_gpu011.py','tests/test_v5_cyr_gpu011_entry.py','-q'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('R1 requires a Colab CUDA GPU')
DEVICE=torch.device('cuda')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from v5_experiments import cyr_gpu011 as base
from anra_v5.cyr_gpu012_r1_run import calibrate_all
from v5_experiments.cyr_gpu012_r1 import resolve_from_calibrations
data=base.load_ark002b_manifest(REPO/'docs/cymek/experiments/CYR-GPU-011/ARK002B_TASK_MANIFEST.json')
assert data['source_split_sha256']==PREREG['data']['split_sha256']
assert data['source_blob_sha']==PREREG['data']['manifest_blob_sha']
CALIBRATIONS=calibrate_all(data=data,torch=torch,device=DEVICE)
Path('/content/CYR-GPU-012-R1-CALIBRATIONS.json').write_text(json.dumps(CALIBRATIONS,indent=2,sort_keys=True))
for k,v in sorted(CALIBRATIONS.items()): print(k,v.get('status'),'updates/s=',round(float(v.get('training_updates_per_sec',0)),3),'rows/s=',round(float(v.get('semantic_rows_per_sec',0)),1),'peak_GB=',round(float(v.get('peak_vram_gb',0)),2))
RESOLVED=resolve_from_calibrations(CALIBRATIONS)
Path('/content/CYR-GPU-012-R1-RESOLVED.json').write_text(json.dumps(RESOLVED,indent=2,sort_keys=True))
print(json.dumps(RESOLVED,indent=2))
print('R1 PREEXECUTION GATE: PASS')


In [ ]:
# CYR-GPU-012 / R1 — CELL 1: RUN WITH 175-MIN HARD WALL + DRIVE ARM REUSE
import json, sys
from pathlib import Path
import torch
from google.colab import drive
drive.mount('/content/drive')
REPO=Path('/content/An-Ra-the-new-AGI'); sys.path.insert(0,str(REPO))
PREREG=json.loads(Path('/content/CYR-GPU-012-R1-PREREGISTRATION.json').read_text())
CALIBRATIONS=json.loads(Path('/content/CYR-GPU-012-R1-CALIBRATIONS.json').read_text())
RESOLVED=json.loads(Path('/content/CYR-GPU-012-R1-RESOLVED.json').read_text())
OUT=Path('/content/drive/MyDrive/CYMEK/CYR-GPU-012-R1'); OUT.mkdir(parents=True,exist_ok=True)
from anra_v5.cyr_gpu012_r1_run import run_campaign
campaign=run_campaign(repo=REPO,out=OUT,preregistration=PREREG,resolved=RESOLVED,calibrations=CALIBRATIONS,torch=torch,device=torch.device('cuda'),progress=lambda m: print(m,flush=True))
print('STATUS:',campaign['status']); print('VERDICT:',campaign.get('decision',{}).get('verdict'))
print('PER-SEED:',json.dumps(campaign.get('decision',{}).get('per_seed',[]),indent=2))
print('BUNDLE:',campaign['bundle']['path'])


In [ ]:
# CYR-GPU-012 / R1 — CELL 2: VERIFY / DOWNLOAD
import hashlib,json
from pathlib import Path
from google.colab import files
OUT=Path('/content/drive/MyDrive/CYMEK/CYR-GPU-012-R1')
receipt=json.loads((OUT/'campaign_receipt.json').read_text())
bundle=Path(receipt['bundle']['path']); assert bundle.exists()
actual=hashlib.sha256(bundle.read_bytes()).hexdigest(); assert actual==receipt['bundle']['sha256']
print('verified:',bundle.name,actual); print('status:',receipt['status']); print('verdict:',receipt.get('decision',{}).get('verdict'))
for row in receipt.get('decision',{}).get('per_seed',[]): print(row)
files.download(str(bundle))
